# MG Score Distributions by Text Length

Compare stored classifier `mg_score` values for human-generated (HG) and machine-generated (MG) texts across token-length bins and feature sets.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import pandas as pd

import seaborn as sns
from IPython.display import display

from notebooks.utils.mg_score_length_distributions_utils import (
    load_mg_score_length_frame,
    plot_mg_score_by_length,
)

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))

## Parameters

Each plot uses one evaluation dataset and classifier. Feature sets are shown as separate panels with shared HG/MG colors and score scale.

In [ ]:
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
SCORER_LLM = "gemma_3_1b_it"
FEATURE_SETS = ("scalar_all", "scalar_compact", "simple_similarity", "extended")
MODEL_FAMILIES = ("logistic_regression", "random_forest")
TRAINING_COMPOSITION = "Kaggle+MAGE+Outfox+BBC"
EVALUATION_COMPOSITIONS = ("kaggle_train", "mage_train", "outfox_chatgpt_train")
PLOT_DATASETS = (*EVALUATION_COMPOSITIONS, "mixed")
TRAINING_WINDOW = 32
LENGTH_BIN_SIZE = 100

## Load Selected Predictions

In [ ]:
score_frame = load_mg_score_length_frame(
    DATA_ROOT,
    run_id=RUN_ID,
    scorer_llm=SCORER_LLM,
    feature_sets=FEATURE_SETS,
    training_window=TRAINING_WINDOW,
    model_families=MODEL_FAMILIES,
    training_composition=TRAINING_COMPOSITION,
    evaluation_compositions=EVALUATION_COMPOSITIONS,
)
print(f"Rows: {len(score_frame):,}")
display(score_frame.groupby(["feature_set", "dataset", "model_family", "label"], observed=True).size().rename("rows").reset_index())

## HG/MG Score Distributions

Each figure is one evaluation dataset and one classifier. Each feature set has its own panel, while HG and MG distributions use the same colors and y-axis scale.

In [ ]:
plot_frame = pd.concat(
    [score_frame, score_frame.assign(dataset="mixed")],
    ignore_index=True,
)

for dataset in PLOT_DATASETS:
    for model_family in MODEL_FAMILIES:
        figure = plot_mg_score_by_length(
            plot_frame,
            dataset=dataset,
            model_family=model_family,
            length_bin_size=LENGTH_BIN_SIZE,
        )
        display(figure)